In [2]:
from __future__ import division
import pandas as pd
import numpy as np
from copy import deepcopy

import warnings
warnings.filterwarnings('ignore')

#!pip install py_stringmatching
#!pip install py_entitymatching
#!pip install py_stringsimjoin
import py_stringmatching as sm
import py_entitymatching as em
import py_stringsimjoin as ssj

from py_entitymatching.catalog import catalog_manager as cm

# Funzioni utilizzate

In [3]:
def ChiusuraRiflessivaTransitiva(MatchTable, ListaID):
  def transitive_closure(a):
    closure = set(a)
    while True:
        new_relations = set((x,w) for x,y in closure for q,w in closure if q == y)
        closure_until_now = closure | new_relations
        if closure_until_now == closure:
            break
        closure = closure_until_now
    return closure


  X=transitive_closure([(x[1],x[0]) for x in MatchTable.values.tolist()]
                        +
                        [(x[0],x[1]) for x in MatchTable.values.tolist()])
  TRANSITIVE_CLOSURE=pd.DataFrame(data=list(X),      columns=['l_id','r_id'])


  REFLEXIVE_CLOSURE=pd.DataFrame({'l_id': ListaID ,
                                  'r_id': ListaID })

  REFLEXIVE_SYMMETRIC_TRANSITIVE_CLOSURE=TRANSITIVE_CLOSURE.append(REFLEXIVE_CLOSURE, ignore_index=True)
  REFLEXIVE_SYMMETRIC_TRANSITIVE_CLOSURE

  return REFLEXIVE_SYMMETRIC_TRANSITIVE_CLOSURE

def CalcoloDeiCluster(MatchTableCRT):
    CLUSTERS = MatchTableCRT.groupby('l_id').agg({'r_id': np.max}).reset_index()
    CLUSTERS.columns=['ClusterElement','ClusterKey']
    CLUSTERS=CLUSTERS[['ClusterKey','ClusterElement']]


    return CLUSTERS.sort_values('ClusterKey')

def CalcolaMatchIndottiCluster(Cluster):
  Join=pd.merge(Cluster,Cluster, on='ClusterKey')
  Join=Join[Join.ClusterElement_x<Join.ClusterElement_y]
  Join=Join[['ClusterElement_x','ClusterElement_y']]
  Join.columns=['l_id','r_id']

  return Join.drop_duplicates()

# in alcuni esercizi è stata usata con un nome diverso:
def MatchIndottiCluster(Cluster):
  Join=pd.merge(Cluster,Cluster, on='ClusterKey')
  Join=Join[Join.ClusterElement_x<Join.ClusterElement_y]
  Join=Join[['ClusterElement_x','ClusterElement_y']]
  Join.columns=['l_id','r_id']

  return Join.drop_duplicates()


In [4]:
def stable_marriage(MatchTable:pd.DataFrame):
    MATCH = pd.DataFrame(columns=['l_id', 'r_id', "sim"])
    MT = deepcopy(MatchTable)
    MT = MT.sort_values(["sim"], ascending=[False])
    while True:
        R = MT.loc[(~MT['l_id'].isin(MATCH['l_id'])) & (~MT['r_id'].isin(MATCH['r_id']))]
        if len(R) == 0:
            break
        x = R.iloc[0,:]
        MATCH = MATCH.append(x, ignore_index=True)
    return MATCH

def simmetric_best_match(MatchTable:pd.DataFrame):
  CMT = deepcopy(MatchTable)

  CMT['A_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['l_id']) \
             .cumcount() + 1

  CMT['B_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['r_id']) \
             .cumcount() + 1

  return CMT[(CMT.A_RowNo==1) & (CMT.B_RowNo==1)].drop(columns=['A_RowNo', 'B_RowNo']).sort_values(['sim'], ascending=[False])

In [5]:
def Valuta(Gold:pd.DataFrame, Match:pd.DataFrame):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]
    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [6]:
def IdSOURCES(Sources:list):
  ListaID= []
  for s in Sources.keys():
    ListaID += Sources[s]['id'].to_list()
  return ListaID

# Esempio



Considerare solo il testo dell'esempio e non lo svolgimento in quanto vengono utilizzate **vecchie** versioni di alcune funzioni, ad esempio **ChiusuraRiflessivaTransitiva** per la clusterizzazione



In [7]:
#
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/Sintetici/'

src_links = [
path+'S1_dirty_.csv',
path+'S2_dirty_.csv',
path+'S3_dirty_.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

src_links = [
path+'S1_clean_.csv',
path+'S2_clean_.csv',
path+'S3_clean_.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

SOURCES['S3']=SOURCES['S3'][SOURCES['S3'].id!='S3_2']

In [9]:
GoldStandardCLEAN=pd.read_csv(path+"GoldStandardClean.csv")
GoldStandardCLEAN


,l_id,r_id,sim
0,S1_8,S2_4,1.000000
1,S1_9,S2_2,0.793103
2,S1_10,S2_7,0.540541
3,S1_6,S2_0,0.531250
4,S1_3,S2_1,0.526316
5,S1_5,S3_0,1.000000
6,S1_8,S3_6,1.000000
7,S1_4,S3_1,0.862069
8,S1_7,S3_7,0.807692
9,S1_0,S3_9,0.733333


In [11]:
ClusterGoldStandardCLEAN = CalcoloDeiCluster(ChiusuraRiflessivaTransitiva(GoldStandardCLEAN,IdSOURCES(SOURCES)))
ClusterGoldStandardCLEAN

,ClusterKey,ClusterElement
1,S1_1,S1_1
4,S1_2,S1_2
13,S2_1,S2_1
5,S2_1,S1_3
15,S2_3,S2_3
2,S2_7,S1_10
19,S2_7,S2_7
7,S3_0,S1_5
20,S3_0,S3_0
6,S3_1,S1_4


In [12]:
# si possono visualizzare i cluster
def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[:2]

      Campi = {
          '#Sources' :     x['source'].nunique(),
          'Sources' :     x['source'].drop_duplicates().str.cat(sep=','),
          '#Elements' :     x['ClusterElement'].nunique(),
          'Elements' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
ClusterGoldStandardCLEAN.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('#Sources', ascending=False)


,ClusterKey,#Sources,Sources,#Elements,Elements
8,S3_3,3,"S3,S2,S1",3,"S3_3,S2_2,S1_9"
10,S3_5,3,"S2,S1,S3",3,"S2_0,S1_6,S3_5"
11,S3_6,3,"S3,S2,S1",3,"S3_6,S2_4,S1_8"
12,S3_7,3,"S2,S1,S3",3,"S2_5,S1_7,S3_7"
2,S2_1,2,"S2,S1",2,"S2_1,S1_3"
4,S2_7,2,"S1,S2",2,"S1_10,S2_7"
5,S3_0,2,"S1,S3",2,"S1_5,S3_0"
6,S3_1,2,"S1,S3",2,"S1_4,S3_1"
7,S3_11,2,"S1,S3",2,"S1_11,S3_11"
9,S3_4,2,"S2,S3",2,"S2_6,S3_4"


In [ ]:
# GoldStandardCLEAN è riferito a tutte e tre le sorgenti
GoldStandardCLEAN.sample(5)

,l_id,r_id,sim
7,S1_4,S3_1,0.862069
12,S1_9,S3_3,0.529412
13,S2_4,S3_6,1.000000
3,S1_6,S2_0,0.531250
5,S1_5,S3_0,1.000000


In [ ]:
# per ottenere quello di una coppia di sorgenti
def GoldStandard(GS,s1,s2):
    return GS[ (GS['l_id'].isin(SOURCES[s1]['id'])) & GS['r_id'].isin(SOURCES[s2]['id'])]

In [ ]:
GoldStandard(GoldStandardCLEAN,'S1','S3')

,l_id,r_id,sim
5,S1_5,S3_0,1.000000
6,S1_8,S3_6,1.000000
7,S1_4,S3_1,0.862069
8,S1_7,S3_7,0.807692
9,S1_0,S3_9,0.733333
10,S1_11,S3_11,0.676471
11,S1_6,S3_5,0.531250
12,S1_9,S3_3,0.529412


## Matching tra una coppia di sources

In [ ]:
# si considera la coppia di sources
A=SOURCES['S1']
B=SOURCES['S3']

A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })

em.set_key(A, 'l_id')
em.set_key(B, 'r_id')

True

In [ ]:
# BLOCKING

A['mix'] = A['given_name'] + ' ' + A['surname'] + ' ' + A['date_of_birth']
B['mix'] = B['given_name'] + ' ' + B['surname'] + ' ' + B['date_of_birth']


C_SimJoin_mix  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                    'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                    l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                    r_out_attrs=['given_name', 'surname', 'date_of_birth'])
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'l_l_id': 'l_id'})
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'r_r_id': 'r_id'})
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'_sim_score': 'sim'})

em.set_key(C_SimJoin_mix, '_id')
em.set_ltable(C_SimJoin_mix, A)
em.set_rtable(C_SimJoin_mix, B)
em.set_fk_ltable(C_SimJoin_mix, 'l_id')
em.set_fk_rtable(C_SimJoin_mix, 'r_id')
C_SimJoin_mix

0% [##########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069
3,3,S1_9,S3_3,michael,lierach,19360816,liersch,michael,19360816,0.529412
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471


In [ ]:
# Matching

In [ ]:
FeatureVector = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)


In [ ]:
# DOMANDA: commentare il significato della seguente rule e valutarne il risultato

brm = em.BooleanRuleMatcher()
brm.add_rule(['surname_surname_exm(ltuple, rtuple) == 1', 'date_of_birth_date_of_birth_lev_dist(ltuple, rtuple) == 1'], FeatureVector)
brm.add_rule(['surname_surname_lev_sim(ltuple, rtuple)*0.5 \
                + given_name_given_name_lev_sim(ltuple, rtuple)*0.2 \
                 + date_of_birth_date_of_birth_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.7'], FeatureVector)
predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
MT

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815,1
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000,1
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069,1
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250,1
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000,1
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692,1
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333,1
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471,1


In [ ]:
Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)


,MT,TP,FP,FN,P,R,F
0,8,7,1,1,0.875,0.875,0.875


In [ ]:
VV=VediValuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FP')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,given_name_y,surname_y,date_of_birth_y,mix_y
0,S1_2,S3_0,right_only,emmerson,lock,19211129,emmerson lock 19211129,emmerson,loyck,19211129,emmerson loyck 19211129


In [ ]:
VV=VediValuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FN')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,given_name_y,surname_y,date_of_birth_y,mix_y
0,S1_9,S3_3,left_only,michael,lierach,19360816,michael lierach 19360816,liersch,michael,19360816,liersch michael 19360816


In [ ]:
# come considerare questo falso negativo dovuto allo scambio tra nome e cognome?
# definire NAME come concatenazione ed usare una jaccard
# ()

In [ ]:
# non è necessario rifare blocking, ma solo definire il nuovo attributo
A['NAME'] = A['given_name'] + ' ' + A['surname']
B['NAME'] = B['given_name'] + ' ' + B['surname']

# e rigenerare le features, verificando le features per il nuovo attributo
FeatureVector = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
#FeatureVector

In [ ]:
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
brm = em.BooleanRuleMatcher()
brm.add_rule(['surname_surname_exm(ltuple, rtuple) == 1', 'date_of_birth_date_of_birth_lev_dist(ltuple, rtuple) == 1'], F)
brm.add_rule(['NAME_NAME_jac_qgm_3_qgm_3(ltuple, rtuple)*0.6 \
                 + date_of_birth_date_of_birth_jac_qgm_3_qgm_3(ltuple, rtuple)*0.4 > 0.5'], F)
predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
MT

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815,1
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000,1
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069,1
3,3,S1_9,S3_3,michael,lierach,19360816,liersch,michael,19360816,0.529412,1
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250,1
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000,1
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692,1
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333,1
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471,1


In [ ]:
Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)

,MT,TP,FP,FN,P,R,F
0,9,8,1,0,0.8889,1.0,0.9412


In [ ]:
VV=VediValuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FP')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,NAME_x,given_name_y,surname_y,date_of_birth_y,mix_y,NAME_y
0,S1_2,S3_0,right_only,emmerson,lock,19211129,emmerson lock 19211129,emmerson lock,emmerson,loyck,19211129,emmerson loyck 19211129,emmerson loyck


In [ ]:
MT=stable_marriage(MT)

In [ ]:
Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)

,MT,TP,FP,FN,P,R,F
0,8,8,0,0,1.0,1.0,1.0


## Matching tra n sorgenti

In [ ]:
def BlockingMatchingRule(A,B):
  A=A.rename(columns={'id': "l_id" })
  B=B.rename(columns={'id': "r_id" })

  A['NAME'] = A['given_name'] + ' ' + A['surname']
  B['NAME'] = B['given_name'] + ' ' + B['surname']

  A['mix'] = A['given_name'] + ' ' + A['surname'] + ' ' + A['date_of_birth']
  B['mix'] = B['given_name'] + ' ' + B['surname'] + ' ' + B['date_of_birth']

  em.set_key(A, 'l_id')
  em.set_key(B, 'r_id')

# BLOCKING
  C_SimJoin_mix  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                    'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                    l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                    r_out_attrs=['given_name', 'surname', 'date_of_birth'])
  C_SimJoin_mix=C_SimJoin_mix.rename(columns={'l_l_id': 'l_id'})
  C_SimJoin_mix=C_SimJoin_mix.rename(columns={'r_r_id': 'r_id'})
  C_SimJoin_mix=C_SimJoin_mix.rename(columns={'_sim_score': 'sim'})

  em.set_key(C_SimJoin_mix, '_id')
  em.set_ltable(C_SimJoin_mix, A)
  em.set_rtable(C_SimJoin_mix, B)
  em.set_fk_ltable(C_SimJoin_mix, 'l_id')
  em.set_fk_rtable(C_SimJoin_mix, 'r_id')
  em.set_fk_rtable(C_SimJoin_mix, 'r_id')

# MATCHING

  F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
  brm = em.BooleanRuleMatcher()
  brm.add_rule(['surname_surname_exm(ltuple, rtuple) == 1',
                'date_of_birth_date_of_birth_lev_dist(ltuple, rtuple) == 1'], F)
  brm.add_rule(['NAME_NAME_jac_qgm_3_qgm_3(ltuple, rtuple)*0.6 \
                 + date_of_birth_date_of_birth_jac_qgm_3_qgm_3(ltuple, rtuple)*0.4 > 0.5'], F)
  predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
  MT=predictions[predictions.pred_label==1]

  return MT

In [ ]:
# che viene applicata a tutte le coppie di sorgenti

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [ ]:
MatchTableSOURCES(SOURCES)

0% [########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,l_id,r_id,sim
0,S1_8,S2_4,1.000000
1,S1_9,S2_2,0.793103
2,S1_10,S2_7,0.540541
3,S1_6,S2_0,0.531250
4,S1_3,S2_1,0.526316
5,S1_5,S2_3,0.500000
0,S1_5,S3_0,1.000000
1,S1_8,S3_6,1.000000
2,S1_4,S3_1,0.862069
3,S1_7,S3_7,0.807692


In [ ]:
# clusterizzazione
MTSOURCES=MatchTableSOURCES(SOURCES)
#MTSOURCES.to_csv("GlobalStandardClean.csv", index=False)

0% [########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


In [ ]:
CRT=ChiusuraRiflessivaTransitiva(MTSOURCES,IdSOURCES(SOURCES))
CRT

,l_id,r_id
0,S1_8,S1_8
1,S1_0,S3_9
2,S3_11,S3_11
3,S3_3,S3_3
4,S3_7,S3_7
...,...,...
94,S3_6,S3_6
95,S3_7,S3_7
96,S3_8,S3_8
97,S3_9,S3_9


In [ ]:
Cluster = CalcoloDeiCluster(CRT)

In [ ]:
# si possono visualizzare i cluster

def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[1]

      Campi = {
          'NumeroSource' :     x['source'].nunique(),
          'QualiSource' :     x['source'].drop_duplicates().str.cat(sep=','),
          'NumeroElementiDistinti' :     x['ClusterElement'].nunique(),
          'ClusterElement' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
Cluster.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('NumeroSource', ascending=False)


,ClusterKey,NumeroSource,QualiSource,NumeroElementiDistinti,ClusterElement
4,S3_0,3,"1,3,2",3,"S1_5,S3_0,S2_3"
7,S3_3,3,"3,2,1",3,"S3_3,S2_2,S1_9"
9,S3_5,3,"2,1,3",3,"S2_0,S1_6,S3_5"
10,S3_6,3,"3,2,1",3,"S3_6,S2_4,S1_8"
11,S3_7,3,"2,1,3",3,"S2_5,S1_7,S3_7"
2,S2_1,2,"2,1",2,"S2_1,S1_3"
3,S2_7,2,"1,2",2,"S1_10,S2_7"
5,S3_1,2,"1,3",2,"S1_4,S3_1"
6,S3_11,2,"1,3",2,"S1_11,S3_11"
8,S3_4,2,"2,3",2,"S2_6,S3_4"


In [ ]:
MatchIndottiCluster=CalcolaMatchIndottiCluster(Cluster)

In [ ]:
ClusterGoldStandardCLEAN = CalcoloDeiCluster(ChiusuraRiflessivaTransitiva(GoldStandardCLEAN,IdSOURCES(SOURCES)))
ClusterGoldStandardCLEAN

,ClusterKey,ClusterElement
1,S1_1,S1_1
4,S1_2,S1_2
13,S2_1,S2_1
5,S2_1,S1_3
15,S2_3,S2_3
2,S2_7,S1_10
19,S2_7,S2_7
7,S3_0,S1_5
20,S3_0,S3_0
6,S3_1,S1_4


In [ ]:
MatchIndottiClusterGoldStandard=CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN)

In [ ]:
Valuta(MatchIndottiClusterGoldStandard,MatchIndottiCluster)

,MT,TP,FP,FN,P,R,F
0,21,19,2,0,0.9048,1.0,0.95


In [ ]:
VV=VediValuta(MatchIndottiClusterGoldStandard,MatchIndottiCluster,'FP')
VV

,l_id,r_id,_merge
19,S1_5,S2_3,right_only
20,S2_3,S3_0,right_only


In [ ]:
SOURCES['S1'][SOURCES['S1'].id=='S1_5']

,given_name,surname,date_of_birth,id
5,emmerson,loyck,19211129,S1_5


In [ ]:
SOURCES['S2'][SOURCES['S2'].id=='S2_3']

,given_name,surname,date_of_birth,id
3,emmeron,loyk,19321129,S2_3


In [ ]:
SOURCES['S3'][SOURCES['S3'].id=='S3_0']

,given_name,surname,date_of_birth,id
0,emmerson,loyck,19211129,S3_0


In [ ]:
# consideriamo le seguenti  Sources ... molto particolari
SOURCES = {}
DF=pd.DataFrame({  'Nome' : [ 'aaaaa','aabbb']})
SOURCES['S1']=DF
DF=pd.DataFrame({  'Nome' : [ 'aaaa']})
SOURCES['S2']=DF
DF=pd.DataFrame({  'Nome' : [ 'aaaabbb']})
SOURCES['S3']=DF


In [ ]:
A=SOURCES['S1']
B=SOURCES['S3']

A=A.rename(columns={'Nome': "l_id" })
B=B.rename(columns={'Nome': "r_id" })

em.set_key(A, 'l_id')
em.set_key(B, 'r_id')

True

In [ ]:
A

,l_id
0,aaaaa
1,aabbb


In [ ]:
# CASO CLEAN
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/NCVR2/'
path=''

src_links = [
path+'NCVR_AF_clean.csv',
path+'NCVR_BF_clean.csv',
path+'NCVR_CF_clean.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandardCLEAN=pd.read_csv(path + "GoldStandardCLEAN2.csv")

In [ ]:
# Consideriamo la loro unione nel dataframe UNIONE
# il cui schema sarà quello di una sorgente (hanno tutti lo stesso schema)
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
# quindi effettuo unione tramite append
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])

In [ ]:
CRT=ChiusuraRiflessivaTransitiva(GoldStandardCLEAN, IdSOURCES(SOURCES))
Cluster = CalcoloDeiCluster(CRT)

def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[:1]
      x['ClusterElement']=x['ClusterElement'].astype(str)

      Campi = {
          '#Sources' :     x['source'].nunique(),
          'Sources' :     x['source'].drop_duplicates().str.cat(sep=','),
          '#Elements' :     x['ClusterElement'].nunique(),
          'Elements' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
Cluster.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('#Elements', ascending=False)


,ClusterKey,#Sources,Sources,#Elements,Elements
243,2_922_12843169,3,"0,1,2",3,"0_922_12748976,1_922_12317422,2_922_12843169"
129,2_1240_13627651,3,"1,2,0",3,"1_1240_1340473,2_1240_13627651,0_1240_1391987"
151,2_222_12748029,3,"1,2,0",3,"1_222_13226442,2_222_12748029,0_222_3198122"
123,2_1022_7999709,3,"2,0,1",3,"2_1022_7999709,0_1022_7968679,1_1022_8529932"
124,2_1040_10986386,3,"2,0,1",3,"2_1040_10986386,0_1040_10986345,1_1040_10950068"
...,...,...,...,...,...
88,1_4541_13015274,1,1,1,1_4541_13015274
90,1_4841_416666,1,1,1,1_4841_416666
91,1_5041_6885017,1,1,1,1_5041_6885017
92,1_5122_3877854,1,1,1,1_5122_3877854


## Premessa

Con il seguente codice si fissano e si verificano le features da usare nel matching

In [ ]:
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
FeaturesList

['first_name_first_name_jac_qgm_3_qgm_3',
 'first_name_first_name_cos_dlm_dc0_dlm_dc0',
 'first_name_first_name_jac_dlm_dc0_dlm_dc0',
 'first_name_first_name_mel',
 'first_name_first_name_lev_dist',
 'first_name_first_name_lev_sim',
 'first_name_first_name_nmw',
 'first_name_first_name_sw',
 'last_name_last_name_jac_qgm_3_qgm_3',
 'last_name_last_name_cos_dlm_dc0_dlm_dc0',
 'last_name_last_name_jac_dlm_dc0_dlm_dc0',
 'last_name_last_name_mel',
 'last_name_last_name_lev_dist',
 'last_name_last_name_lev_sim',
 'last_name_last_name_nmw',
 'last_name_last_name_sw',
 'sex_sex_lev_dist',
 'sex_sex_lev_sim',
 'sex_sex_jar',
 'sex_sex_jwn',
 'sex_sex_exm',
 'sex_sex_jac_qgm_3_qgm_3',
 'age_age_lev_dist',
 'age_age_lev_sim',
 'age_age_jar',
 'age_age_jwn',
 'age_age_exm',
 'age_age_jac_qgm_3_qgm_3',
 'birth_place_birth_place_jac_qgm_3_qgm_3',
 'birth_place_birth_place_cos_dlm_dc0_dlm_dc0',
 'birth_place_birth_place_jac_dlm_dc0_dlm_dc0',
 'birth_place_birth_place_mel',
 'birth_place_birth_place_

In [ ]:
# Features da considerare nel matching
FixedFeatures = F[F.feature_name.isin(['last_name_last_name_lev_sim',
                                       'zip_code_zip_code_exm',
                                       'first_name_first_name_lev_sim'])]

## Domanda

Partendo dalla regola data nella seguente funzione BlockingMatchinRule, dalla visualizzazione e valutazione dei relativi cluster, fare e commentare opportune modifiche alla funzione BlockingMatchinRule per migliorare precizione e recall. Si possono considerare  solo le FixedFeatures o eventualmente anche le altre disponibili

In [ ]:
def BlockingMatchinRule(A,B):
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['first_name'] + ' ' + A['last_name']
    B['mix'] = B['first_name'] + ' ' + B['last_name']

    C_SimJoin_mix  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code'],
                                        r_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code']
                                     )
    C_SimJoin_mix=C_SimJoin_mix.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix=C_SimJoin_mix.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix=C_SimJoin_mix.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix, '_id')
    em.set_ltable(C_SimJoin_mix, A)
    em.set_rtable(C_SimJoin_mix, B)
    em.set_fk_ltable(C_SimJoin_mix, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix, 'r_id')
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['last_name_last_name_lev_sim(ltuple, rtuple) >= .7','zip_code_zip_code_exm(ltuple, rtuple) == 1'], FixedFeatures)
    brm.add_rule([ 'last_name_last_name_lev_sim(ltuple, rtuple) >= .3',
                  'first_name_first_name_lev_sim(ltuple, rtuple) >= .3' ], FixedFeatures)
    predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

#    MT=C_SimJoin_mix

    return MT

In [ ]:
CRT=ChiusuraRiflessivaTransitiva(MatchTableSOURCES(SOURCES),IdSOURCES(SOURCES))
Cluster = CalcoloDeiCluster(CRT)

def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[:2]
      x['ClusterElement']=x['ClusterElement'].astype(str)

      Campi = {
          '#Sources' :     x['source'].nunique(),
          'Sources' :     x['source'].drop_duplicates().str.cat(sep=','),
          '#Elements' :     x['ClusterElement'].nunique(),
          'Elements' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
Cluster.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('#Elements', ascending=False)

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,ClusterKey,#Sources,Sources,#Elements,Elements
152,2_2240_934374,3,"2_,1_,0_",4,"2_2240_934374,2_1122_68628,1_1122_9842647,0_2240_11945491"
243,2_922_12843169,3,"0_,1_,2_",3,"0_922_12748976,1_922_12317422,2_922_12843169"
124,2_1022_7999709,3,"2_,0_,1_",3,"2_1022_7999709,0_1022_7968679,1_1022_8529932"
167,2_2940_9137590,3,"0_,1_,2_",3,"0_3822_11913154,1_3822_11891631,2_2940_9137590"
151,2_222_12748029,3,"2_,1_,0_",3,"2_222_12748029,1_222_13226442,0_222_3198122"
...,...,...,...,...,...
56,1_1941_7943494,1,1_,1,1_1941_7943494
134,2_1431_13734344,1,2_,1,2_1431_13734344
136,2_1522_8428365,1,2_,1,2_1522_8428365
137,2_1531_13717497,1,2_,1,2_1531_13717497


In [ ]:
ClusterGoldStandardCLEAN = CalcoloDeiCluster(ChiusuraRiflessivaTransitiva(GoldStandardCLEAN,IdSOURCES(SOURCES)))
Cluster = CalcoloDeiCluster(ChiusuraRiflessivaTransitiva(MatchTableSOURCES(SOURCES),IdSOURCES(SOURCES)))
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN) , CalcolaMatchIndottiCluster(Cluster))

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,MT,TP,FP,FN,P,R,F
0,107,91,16,11,0.8505,0.8922,0.8708


In [ ]:
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN) ,
              CalcolaMatchIndottiCluster(Cluster),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,0_5640_563518,1_5640_3369336,left_only,0_5640_563518,amanda,conard,female,34,nc,28804,1_5640_3369336,amanda,waldrop,female,34,,27712
1,0_1140_6460300,2_1140_12513318,left_only,0_1140_6460300,amber,broach,female,32,sc,28115,2_1140_12513318,amber,smith,female,32,sc,27613
2,1_1240_1340473,2_1240_13627651,left_only,1_1240_1340473,teresa,pender,female,63,nc,28570,2_1240_13627651,teresa,starnes,female,63,nc,27562
3,0_1240_1391987,1_1240_1340473,left_only,0_1240_1391987,teresa,starnes,female,63,nc,28570,1_1240_1340473,teresa,pender,female,63,nc,28570
4,1_1340_3795261,2_1340_13807668,left_only,1_1340_3795261,denise,hines,female,37,,27801,2_1340_13807668,denise,wells,female,37,nc,27530
5,0_1740_13855498,2_1740_7342937,left_only,0_1740_13855498,kathren,woodham,female,38,,27530,2_1740_7342937,kathren,hayes,female,38,fl,28209
6,1_2622_9753093,2_2622_148884,left_only,1_2622_9753093,briana,lefler,female,26,nc,27231,2_2622_148884,briana,orr,female,26,nc,27253
7,1_2722_10836165,2_2722_8809482,left_only,1_2722_10836165,christine,jones,female,36,nc,28347,2_2722_8809482,christine,horton,female,36,nc,28373
8,0_3140_907191,2_3140_1207977,left_only,0_3140_907191,angel,johnson,female,41,,28690,2_3140_1207977,angel,mcclellan,female,41,nc,28630
9,1_5222_3277797,2_5222_3462830,left_only,1_5222_3277797,kenndra,quiles,female,44,nj,27707,2_5222_3462830,kenndra,johnson,female,44,nj,27707


In [ ]:
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN) ,
              CalcolaMatchIndottiCluster(Cluster),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,0_3551_2874784,1_1741_7343057,right_only,0_3551_2874784,christopher,hines,male,21,nc,27360,1_1741_7343057,christopher,miller,male,29,ga,28227
1,0_3751_1179043,1_5722_659298,right_only,0_3751_1179043,james,kelly,male,68,mi,28025,1_5722_659298,james,keefer,male,52,in,28748
2,1_6141_7624860,2_1840_11606714,right_only,1_6141_7624860,joseph,blumberg,male,38,nc,28269,2_1840_11606714,joseph,baker,male,46,va,28129
3,0_1840_7447052,1_6141_7624860,right_only,0_1840_7447052,joseph,baker,male,46,,28270,1_6141_7624860,joseph,blumberg,male,38,nc,28269
4,2_1122_68628,2_2240_934374,right_only,2_1122_68628,patricia,kurt,female,58,ny,27215,2_2240_934374,latricia,hunter,female,46,nc,28075
5,1_1122_9842647,2_2240_934374,right_only,1_1122_9842647,patricia,kurt,female,58,ny,27517,2_2240_934374,latricia,hunter,female,46,nc,28075
6,0_2240_11945491,2_1122_68628,right_only,0_2240_11945491,latricia,hunter,female,46,,28173,2_1122_68628,patricia,kurt,female,58,ny,27215
7,0_2240_11945491,1_1122_9842647,right_only,0_2240_11945491,latricia,hunter,female,46,,28173,1_1122_9842647,patricia,kurt,female,58,ny,27517
8,1_3541_3168895,2_231_9728052,right_only,1_3541_3168895,william,mcghee,male,76,nc,27703,2_231_9728052,william,mcinerney,male,29,nc,27516
9,0_3822_11913154,2_2940_9137590,right_only,0_3822_11913154,david,troy,male,73,nc,28337,2_2940_9137590,david,brown,male,51,il,28409


In [ ]:
###### INSERIRE LA VOSTRA RISPOSTA A PARTIRE DA QUESTO PUNTO

## Domanda

Confrontare quanto fatto in precedenza nel caso clean con quello che si ottiene nel caso dirty: ripetere le stesse funzioni e fare opportune considerazioni

In [ ]:
# DIRTY
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/NCVR2/'
path=''

src_links = [
path+'NCVR_AF.csv',
path+'NCVR_BF.csv',
path+'NCVR_CF.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }
GoldStandardDIRTY=pd.read_csv(path + "GoldStandardDIRTY2.csv")


In [ ]:
# Consideriamo la loro unione nel dataframe UNIONE
# il cui schema sarà quello di una sorgente (hanno tutti lo stesso schema)
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
# quindi effettuo unione tramite append
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])

In [ ]:
CRT=ChiusuraRiflessivaTransitiva(GoldStandardDIRTY, IdSOURCES(SOURCES))
Cluster = CalcoloDeiCluster(CRT)

def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[:1]
      x['ClusterElement']=x['ClusterElement'].astype(str)

      Campi = {
          '#Sources' :     x['source'].nunique(),
          'Sources' :     x['source'].drop_duplicates().str.cat(sep=','),
          '#Elements' :     x['ClusterElement'].nunique(),
          'Elements' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
Cluster.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('#Elements', ascending=False)


,ClusterKey,#Sources,Sources,#Elements,Elements
153,2_22_9930113,2,"2,0",5,"2_22_3326652,2_22_91220,0_22_3235320,0_22_9865350,2_22_9930113"
79,1_40_13913995,2,"0,1",5,"0_40_12768214,0_40_13809878,0_40_13867121,0_40_9891855,1_40_13913995"
151,2_222_12748029,3,"1,0,2",4,"1_222_13226442,0_222_1749957,0_222_3198122,2_222_12748029"
43,1_122_9000404,2,"1,0",4,"1_122_2397713,1_122_12738159,0_122_9112102,1_122_9000404"
157,2_240_13265262,3,"1,0,2",4,"1_240_116265,1_240_13225092,0_240_94472,2_240_13265262"
...,...,...,...,...,...
65,1_3041_12401522,1,1,1,1_3041_12401522
66,1_3141_13928423,1,1,1,1_3141_13928423
68,1_3241_10003710,1,1,1,1_3241_10003710
71,1_341_6619987,1,1,1,1_341_6619987


In [ ]:
###### INSERIRE LA VOSTRA RISPOSTA A PARTIRE DA QUESTO PUNTO